In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

In [ ]:
file_path = '../../tests/data_/returns.csv'
returns = pd.read_csv(file_path, index_col=0, parse_dates=True)

print(returns.shape)

In [ ]:
start_date = "2021-04-07"
end_date = "2026-04-07"

sp500_tickers = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'BRK-B', 'LLY', 'AVGO', 'TSLA',
    'WMT', 'JPM', 'UNH', 'XOM', 'V', 'MA', 'JNJ', 'PG', 'ORCL', 'HD',
    'COST', 'ABBV', 'BAC', 'NFLX', 'CVX', 'AMD', 'KO', 'CAT', 'PLTR', 'MU'
]
crypto_tickers = [
    'BTC-USD', 'ETH-USD', 'USDT-USD', 'BNB-USD', 'XRP-USD', 
    'USDC-USD', 'SOL-USD', 'TRX-USD', 'DOGE-USD', 'ADA-USD'
]

print(f"[{start_date} ~ {end_date}] 데이터를 다운로드합니다...")
stock_data = yf.download(sp500_tickers, start=start_date, end=end_date, interval='1d', auto_adjust=True)['Close']
crypto_data = yf.download(crypto_tickers, start=start_date, end=end_date, interval='1d', auto_adjust=True)['Close']

[***                    7%                       ]  2 of 30 completed

[2021-04-07 ~ 2026-04-07] 데이터를 다운로드합니다...


[*********************100%***********************]  30 of 30 completed
[*********************100%***********************]  10 of 10 completed


In [27]:
df_final = pd.concat([stock_data, crypto_data], axis=1)
# 주식 불러온 후 합치기
returns = df_final.ffill().pct_change().dropna()
crypto_returns = returns[crypto_tickers]
stock_returns = returns[sp500_tickers]
crypto_medians = crypto_returns.median(axis=1)
stock_medians = stock_returns.median(axis=1)
targets_crypto = crypto_returns.apply(lambda x: (x >= crypto_medians).astype(int))
targets_stock = stock_returns.apply(lambda x: (x >= stock_medians).astype(int))
targets = pd.concat([targets_stock, targets_crypto], axis=1)


C:\Users\andre\AppData\Local\Temp\ipykernel_31972\103556742.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_final = pd.concat([stock_data, crypto_data], axis=1)


In [28]:
def prepare_lstm_data(returns_df, targets_df, stock_tickers, crypto_tickers, train_days=456, test_days=152, seq_length=7):
    X_train_list, y_train_list = [], []
    X_test_list, y_test_list = [], []
    
    total_days = train_days + test_days
    study_data = returns_df.iloc[:total_days]
    study_target = targets_df.iloc[:total_days]
    
    # ==============================================================
    # 1. 주식 / 코인 분리하여 정규화 (Look-ahead bias 방지를 위해 Train 구간만 사용)
    # ==============================================================
    train_data = study_data.iloc[:train_days]
    
    # 주식 시장 통계치 및 정규화
    stock_train = train_data[stock_tickers]
    stock_mean = stock_train.values.mean()
    stock_std = stock_train.values.std()
    norm_stocks = (study_data[stock_tickers] - stock_mean) / stock_std
    
    # 코인 시장 통계치 및 정규화
    crypto_train = train_data[crypto_tickers]
    crypto_mean = crypto_train.values.mean()
    crypto_std = crypto_train.values.std()
    norm_crypto = (study_data[crypto_tickers] - crypto_mean) / crypto_std
    
    # 정규화된 데이터 다시 하나로 합치기
    norm_df = pd.concat([norm_stocks, norm_crypto], axis=1)
    
    # ==============================================================
    # 2. 슬라이딩 윈도우 기반 시퀀스 텐서 생성
    # ==============================================================
    for ticker in norm_df.columns:
        norm_values = norm_df[ticker].values
        target_values = study_target[ticker].values
        
        for j in range(len(norm_values) - seq_length):
            window = norm_values[j : j + seq_length]
            target = target_values[j + seq_length]
            
            if (j + seq_length) < train_days:
                X_train_list.append(window)
                y_train_list.append(target)
            else:
                X_test_list.append(window)
                y_test_list.append(target)
                
    # 3. 3차원 텐서 반환
    X_train = np.expand_dims(np.array(X_train_list), -1)
    X_test = np.expand_dims(np.array(X_test_list), -1)
    y_train = np.array(y_train_list)
    y_test = np.array(y_test_list)
    
    # norm_df 도 같이 반환하도록 수정
    return X_train, y_train, X_test, y_test, norm_df

# ==========================================
# 실행 및 중간 확인
# ==========================================
# 받을 때 변수를 하나 더(norm_df) 써줍니다.
X_train, y_train, X_test, y_test, norm_df = prepare_lstm_data(
    returns, 
    targets, 
    sp500_tickers,   
    crypto_tickers   
)

# 사진처럼 가로세로를 눕혀서 확인
visual_df = norm_df.T
visual_df.columns = visual_df.columns.strftime('%Y-%m-%d') # 날짜 깔끔하게

print("========== [정규화 완료 데이터 2D 표 확인] ==========")
display(visual_df.head(5).iloc[:, :10])

========== [정규화 완료 데이터 2D 표 확인] ==========


Date,2021-04-08,2021-04-09,2021-04-10,2021-04-11,2021-04-12,2021-04-13,2021-04-14,2021-04-15,2021-04-16,2021-04-17
Ticker,,,,,,,,,,
NVDA,0.660776,0.308944,-0.006308,-0.006308,3.048777,1.675964,-1.402252,3.055822,-0.763683,-0.006308
AAPL,1.039621,1.094963,-0.006308,-0.006308,-0.725910,1.315460,-0.977156,1.011017,-0.143750,-0.006308
MSFT,0.722675,0.551984,-0.006308,-0.006308,0.006447,0.541909,-0.616383,0.825599,0.253537,-0.006308
AMZN,0.323844,1.195245,-0.006308,-0.006308,0.109636,0.325340,-1.077911,0.745677,0.321185,-0.006308
GOOGL,0.270556,0.482776,-0.006308,-0.006308,-0.630167,0.231349,-0.308309,1.044947,-0.065796,-0.006308


In [ ]:
# ==============================================================
# 전체 기간 Walk-Forward Splits 자동 생성 함수
# ==============================================================
def create_all_walk_forward_splits(returns, targets, stock_tickers, crypto_tickers, train_days=456, test_days=152, seq_length=30, step_size=152):
    splits = []
    total_days = train_days + test_days
    
    # 1. 현재 가진 데이터로 몇 개의 Split을 만들 수 있는지 자동 계산
    max_splits = (len(returns) - total_days) // step_size + 1
    
    print(f"========== 총 {max_splits}개의 Walk-Forward Splits 생성을 시작합니다 ==========")
    
    for i in range(max_splits):
        # 2. 각 Split의 시작 인덱스와 종료 인덱스 계산
        start_idx = i * step_size
        end_idx = start_idx + total_days
        
        # 3. 전체 데이터에서 해당 기간만큼 표 잘라내기
        split_returns = returns.iloc[start_idx:end_idx]
        split_targets = targets.iloc[start_idx:end_idx]
        
        # 4. 방금 만드신 함수를 호출하여 텐서로 변환
        X_train, y_train, X_test, y_test = prepare_lstm_data(
            split_returns, 
            split_targets, 
            stock_tickers,       # 주식 종목 리스트 
            crypto_tickers,      # 코인 종목 리스트
            train_days=train_days, 
            test_days=test_days, 
            seq_length=seq_length
        )
        
        # 5. 완성된 텐서를 리스트에 차곡차곡 보관
        splits.append((X_train, y_train, X_test, y_test))
        
        # 6. 원하시는 형식으로 출력
        print(f"✅ Split {i+1} 생성 완료: {start_idx}일 ~ {end_idx}일 (Train {len(X_train)}개, Test {len(X_test)}개)")
        
    print("========== 모든 기간의 데이터 준비가 완료되었습니다 ==========")
    return splits

# --- 실행 ---
# (sp500_tickers와 crypto_tickers는 다운로드 시 사용했던 변수명을 넣어주세요)
all_splits = create_all_walk_forward_splits(
    returns, 
    targets, 
    sp500_tickers, 
    crypto_tickers, 
    train_days=456, 
    test_days=152, 
    seq_length=7, 
    step_size=152
)

In [29]:
def get_ticker_2d_table(norm_df, target_df, ticker, seq_length=7):
    # 1. 특정 종목의 정규화 데이터와 정답지 추출
    ticker_norm = norm_df[ticker].values
    ticker_target = target_df[ticker].values
    dates = norm_df.index
    
    data_list = []
    
    # 2. 7일 단위 슬라이딩 윈도우 진행
    for i in range(len(ticker_norm) - seq_length):
        # t-7 부터 t-1 까지의 7일치 흐름
        window = ticker_norm[i : i + seq_length]
        
        # t일(target_date)의 정답
        tgt = ticker_target[i + seq_length]
        
        # target_date 날짜 문자열 변환
        target_date = dates[i + seq_length].strftime('%Y-%m-%d')
        
        # 3. 한 줄(Row)로 합치기: [날짜, t-7, t-6, ..., t-1, 정답]
        row = [target_date] + list(window) + [int(tgt)]
        data_list.append(row)
        
    # 4. 컬럼명 생성 및 데이터프레임 변환
    columns = ['target_date'] + [f't-{seq_length - j}' for j in range(seq_length)] + ['target']
    df_2d = pd.DataFrame(data_list, columns=columns)
    
    return df_2d

# ==========================================
# 실행 및 확인
# ==========================================

target_ticker = 'NVDA' 

nvda_table = get_ticker_2d_table(norm_df, targets, target_ticker, seq_length=7)

print(f"========== [{target_ticker}] 7일 시퀀스 2D 데이터표 ==========")
display(nvda_table.head())

========== [NVDA] 7일 시퀀스 2D 데이터표 ==========


,target_date,t-7,t-6,t-5,t-4,t-3,t-2,t-1,target
0,2021-04-15,0.660776,0.308944,-0.006308,-0.006308,3.048777,1.675964,-1.402252,1
1,2021-04-16,0.308944,-0.006308,-0.006308,3.048777,1.675964,-1.402252,3.055822,0
2,2021-04-17,-0.006308,-0.006308,3.048777,1.675964,-1.402252,3.055822,-0.763683,1
3,2021-04-18,-0.006308,3.048777,1.675964,-1.402252,3.055822,-0.763683,-0.006308,1
4,2021-04-19,3.048777,1.675964,-1.402252,3.055822,-0.763683,-0.006308,-0.006308,0


In [30]:
import torch
import torch.nn as nn

class WalkForwardLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super(WalkForwardLSTM, self).__init__()
        
        # LSTM 레이어 설정
        # input_size=1 : 우리는 '수익률'이라는 1개의 지표(Feature)만 사용합니다.
        # batch_first=True : 텐서의 첫 번째 차원이 '배치(샘플 수)'임을 명시합니다.
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        
        # 완전 연결층 (Fully Connected Layer)
        # LSTM이 추출한 64개의 특징을 바탕으로 최종 1개의 결괏값을 뽑아냅니다.
        self.fc = nn.Linear(hidden_size, 1)
        
        # 활성화 함수 (Sigmoid)
        # 0과 1 (상대적 하락/상승) 분류 문제이므로 결괏값을 0~1 사이의 확률로 압축합니다.
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. 데이터를 LSTM에 통과시킵니다.
        # x의 형태: (Batch Size, 7일, 1)
        lstm_out, _ = self.lstm(x)
        
        # 2. 7일 치의 흐름 중 '마지막 7일째'가 판단한 최종 결과만 가져옵니다.
        # lstm_out[:, -1, :] 의미: 모든 배치에 대해(:), 마지막 시퀀스(-1)의, 모든 은닉상태(:)
        last_time_step = lstm_out[:, -1, :]
        
        # 3. 선형 층과 시그모이드 함수를 통과시켜 0~1 사이 확률값을 얻습니다.
        out = self.fc(last_time_step)
        return self.sigmoid(out)

# 모델 생성 및 디바이스(CPU/GPU) 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WalkForwardLSTM().to(device)

print(f"========== LSTM 모델 생성 완료 ==========")
print(f"사용 중인 디바이스: {device}")
print(model)

========== LSTM 모델 생성 완료 ==========
사용 중인 디바이스: cpu
WalkForwardLSTM(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [32]:
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# (데이터 로더 준비 과정은 동일합니다)
X_train_1, y_train_1, X_test_1, y_test_1 = all_splits[0]
X_train_t = torch.FloatTensor(X_train_1).to(device)
y_train_t = torch.FloatTensor(y_train_1).view(-1, 1).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

criterion = nn.BCELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001) 
num_epochs = 50 

print(f"========== [Split 1] LSTM 모델 학습 (정확도 포함) ==========")
model.train() 

for epoch in range(num_epochs):
    epoch_loss = 0
    correct_predictions = 0 # 맞춘 문제 수를 저장할 변수 추가!
    total_samples = 0       # 전체 문제 수를 저장할 변수 추가!
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # --- [추가된 부분] 정확도 계산 로직 ---
        # 예측 확률이 0.5 이상이면 1.0(True), 아니면 0.0(False)으로 변환
        predicted_classes = (predictions >= 0.5).float() 
        
        # 내 예측과 실제 정답이 똑같은 것만 개수를 셉니다.
        correct_predictions += (predicted_classes == batch_y).sum().item()
        # 이번 배치에서 푼 총 문제 수를 더합니다.
        total_samples += batch_y.size(0)
        
    # 10번(Epoch)마다 평균 성적표(Loss)와 정확도(Accuracy) 출력
    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(train_loader)
        # 정답률 = (맞춘 개수 / 전체 문제 수) * 100
        accuracy = (correct_predictions / total_samples) * 100 
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("========== [Split 1] 학습 및 채점이 완료되었습니다! ==========")

========== [Split 1] LSTM 모델 학습 (정확도 포함) ==========
Epoch [1/50], Loss: 0.5443, Accuracy: 69.45%
Epoch [10/50], Loss: 0.5136, Accuracy: 71.70%
Epoch [20/50], Loss: 0.4796, Accuracy: 74.19%
Epoch [30/50], Loss: 0.4522, Accuracy: 75.91%
Epoch [40/50], Loss: 0.4225, Accuracy: 77.87%
Epoch [50/50], Loss: 0.4036, Accuracy: 79.19%
========== [Split 1] 학습 및 채점이 완료되었습니다! ==========


In [33]:
# 1. 모델을 '평가 모드'로 전환 (Dropout 등을 끄고 온전한 실력 발휘)
model.eval()

print("========== [Split 1] 실전 모의고사 채점 중... ==========")

# 2. 역전파(학습) 정지: 실전에서는 정답을 보고 배우면 안 되므로 학습 기능을 끕니다.
with torch.no_grad():
    # 테스트 문제지(X_test_1)를 모델에 던져주고 예측을 받습니다.
    test_X_tensor = torch.FloatTensor(X_test_1).to(device)
    test_y_tensor = torch.FloatTensor(y_test_1).view(-1, 1).to(device)
    
    test_predictions = model(test_X_tensor)
    
    # 확률값이 0.5 이상이면 1, 미만이면 0으로 변환
    test_classes = (test_predictions >= 0.5).float()
    
    # 실제 정답과 비교하여 맞춘 개수 계산
    correct_test = (test_classes == test_y_tensor).sum().item()
    total_test = test_y_tensor.size(0)
    
    # 최종 테스트 정확도 도출
    test_accuracy = (correct_test / total_test) * 100

print(f"▶ 실전 테스트 문제 수: 총 {total_test}개")
print(f"▶ 맞춘 문제 수: {correct_test}개")
print(f"🏆 최종 실전 정확도(Test Accuracy): {test_accuracy:.2f}%")

========== [Split 1] 실전 모의고사 채점 중... ==========
▶ 실전 테스트 문제 수: 총 6080개
▶ 맞춘 문제 수: 3232개
🏆 최종 실전 정확도(Test Accuracy): 53.16%


In [34]:
import numpy as np

# 결과를 저장할 리스트
test_accuracies = []

print("========== 🚀 전 구간(Split 1~9) Walk-Forward 백테스트 시작 ==========")

# 9개의 Split을 하나씩 꺼내서 학습하고 평가합니다.
for i, split_data in enumerate(all_splits):
    print(f"\n--- [Split {i+1}] 진행 중... ---")
    X_train, y_train, X_test, y_test = split_data
    
    # 데이터 텐서 변환
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)
    y_test_t = torch.FloatTensor(y_test).view(-1, 1).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # ⚠️ 매 Split마다 모델의 '뇌'를 초기화해야 합니다. (이전 기간의 기억 지우기)
    model = WalkForwardLSTM().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 1. 모델 학습 (로그 출력은 생략하고 빠르게 50 Epoch 진행)
    model.train()
    for epoch in range(50):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            
    # 2. 실전 모의고사 (Test)
    model.eval()
    with torch.no_grad():
        test_predictions = model(X_test_t)
        test_classes = (test_predictions >= 0.5).float()
        correct_test = (test_classes == y_test_t).sum().item()
        total_test = y_test_t.size(0)
        
        test_acc = (correct_test / total_test) * 100
        test_accuracies.append(test_acc)
        
    print(f"✅ Split {i+1} 실전 정확도: {test_acc:.2f}%")

# 3. 최종 결과 요약
print("\n========================================================")
print("🏆 [최종 백테스트 결과 요약]")
for i, acc in enumerate(test_accuracies):
    print(f" - Split {i+1}: {acc:.2f}%")
print(f"🌟 평균 실전 정확도(Average Test Accuracy): {np.mean(test_accuracies):.2f}%")
print("========================================================")

========== 🚀 전 구간(Split 1~9) Walk-Forward 백테스트 시작 ==========

--- [Split 1] 진행 중... ---
✅ Split 1 실전 정확도: 53.96%

--- [Split 2] 진행 중... ---
✅ Split 2 실전 정확도: 54.87%

--- [Split 3] 진행 중... ---
✅ Split 3 실전 정확도: 54.21%

--- [Split 4] 진행 중... ---
✅ Split 4 실전 정확도: 57.29%

--- [Split 5] 진행 중... ---
✅ Split 5 실전 정확도: 58.77%

--- [Split 6] 진행 중... ---
✅ Split 6 실전 정확도: 58.49%

--- [Split 7] 진행 중... ---
✅ Split 7 실전 정확도: 56.20%

--- [Split 8] 진행 중... ---
✅ Split 8 실전 정확도: 60.48%

--- [Split 9] 진행 중... ---
✅ Split 9 실전 정확도: 59.31%

🏆 [최종 백테스트 결과 요약]
 - Split 1: 53.96%
 - Split 2: 54.87%
 - Split 3: 54.21%
 - Split 4: 57.29%
 - Split 5: 58.77%
 - Split 6: 58.49%
 - Split 7: 56.20%
 - Split 8: 60.48%
 - Split 9: 59.31%
🌟 평균 실전 정확도(Average Test Accuracy): 57.06%


In [35]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader

# ==============================================================
# 1. 딥러닝(DNN) 모델 설계 (시간의 흐름을 무시하는 단순 신경망)
# ==============================================================
class SimpleDNN(nn.Module):
    def __init__(self, input_size=7): # 사용자님의 시퀀스 길이 7일에 맞춤
        super(SimpleDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.net(x)

# ==============================================================
# 2. 3개 모델 Walk-Forward 평가 루프
# ==============================================================
results = {'LOG': [], 'RF': [], 'DNN': []}
print("========== 🚀 비교 모델(LOG, RF, DNN) 백테스트 시작 ==========")

for i, split_data in enumerate(all_splits):
    print(f"\n--- [Split {i+1}] 학습 및 평가 중... ---")
    X_train, y_train, X_test, y_test = split_data
    
    # 🌟 핵심 전처리: 3차원 텐서 (Samples, 7, 1) -> 2차원 평면 (Samples, 7) 로 누르기
    X_train_2d = X_train.reshape(X_train.shape[0], -1)
    X_test_2d = X_test.reshape(X_test.shape[0], -1)
    
    # --------------------------------------------------
    # 모델 A: 로지스틱 회귀 (LOG)
    # --------------------------------------------------
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_train_2d, y_train)
    log_preds = log_model.predict(X_test_2d)
    log_acc = accuracy_score(y_test, log_preds) * 100
    results['LOG'].append(log_acc)
    
    # --------------------------------------------------
    # 모델 B: 랜덤 포레스트 (RF)
    # --------------------------------------------------
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train_2d, y_train)
    rf_preds = rf_model.predict(X_test_2d)
    rf_acc = accuracy_score(y_test, rf_preds) * 100
    results['RF'].append(rf_acc)
    
    # --------------------------------------------------
    # 모델 C: 심층 신경망 (DNN)
    # --------------------------------------------------
    dnn_model = SimpleDNN(input_size=7).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(dnn_model.parameters(), lr=0.001)
    
    X_train_t = torch.FloatTensor(X_train_2d).to(device)
    y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_test_t = torch.FloatTensor(X_test_2d).to(device)
    y_test_t = torch.FloatTensor(y_test).view(-1, 1).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    dnn_model.train()
    for epoch in range(30): # 비교군이므로 30 에포크만 빠르게 진행
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            pred = dnn_model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            
    dnn_model.eval()
    with torch.no_grad():
        dnn_preds = dnn_model(X_test_t)
        dnn_classes = (dnn_preds >= 0.5).float()
        correct = (dnn_classes == y_test_t).sum().item()
        dnn_acc = (correct / y_test_t.size(0)) * 100
        results['DNN'].append(dnn_acc)
        
    print(f"✅ LOG: {log_acc:.2f}% | RF: {rf_acc:.2f}% | DNN: {dnn_acc:.2f}%")

# ==============================================================
# 3. 최종 비교 결과 출력
# ==============================================================
print("\n========================================================")
print("🏆 [최종 비교 모델 백테스트 결과 요약 (평균 정확도)]")
print(f" 1. Logistic Regression (LOG): {np.mean(results['LOG']):.2f}%")
print(f" 2. Random Forest (RF)       : {np.mean(results['RF']):.2f}%")
print(f" 3. Deep Neural Network (DNN): {np.mean(results['DNN']):.2f}%")
print("========================================================")

========== 🚀 비교 모델(LOG, RF, DNN) 백테스트 시작 ==========

--- [Split 1] 학습 및 평가 중... ---
✅ LOG: 54.79% | RF: 54.23% | DNN: 55.08%

--- [Split 2] 학습 및 평가 중... ---
✅ LOG: 57.91% | RF: 55.90% | DNN: 57.68%

--- [Split 3] 학습 및 평가 중... ---
✅ LOG: 59.13% | RF: 54.57% | DNN: 58.36%

--- [Split 4] 학습 및 평가 중... ---
✅ LOG: 62.15% | RF: 57.88% | DNN: 61.91%

--- [Split 5] 학습 및 평가 중... ---
✅ LOG: 60.81% | RF: 59.82% | DNN: 61.60%

--- [Split 6] 학습 및 평가 중... ---
✅ LOG: 59.70% | RF: 59.47% | DNN: 60.31%

--- [Split 7] 학습 및 평가 중... ---
✅ LOG: 59.39% | RF: 58.44% | DNN: 58.83%

--- [Split 8] 학습 및 평가 중... ---
✅ LOG: 61.00% | RF: 60.20% | DNN: 61.79%

--- [Split 9] 학습 및 평가 중... ---
✅ LOG: 60.84% | RF: 60.00% | DNN: 61.25%

🏆 [최종 비교 모델 백테스트 결과 요약 (평균 정확도)]
 1. Logistic Regression (LOG): 59.52%
 2. Random Forest (RF)       : 57.83%
 3. Deep Neural Network (DNN): 59.65%


* Rank,Model,Mean Test Accuracy,Feature
* 1st,DNN (Deep Neural Network),59.65%,단순 비선형 패턴 추출에 최적
* 2nd,LOG (Logistic Regression),59.52%,선형적 추세 파악에 매우 강력
* 3rd,RF (Random Forest),57.83%,데이터의 비결정적 구조 학습
* 4th,LSTM (Long Short-Term Memory),56.24%,시퀀스 문맥 및 흐름 파악